In [28]:
import os, sys, json
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from mp_api.client import MPRester 
TEST_RUN = False
MIN_GAP = 0.01
MAX_GAP = 5.0
CHUNK_SIZE = 1000

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
DATA_DIR = NOTEBOOK_DIR.parent / "data"
DATA_DIR.mkdir(exist_ok=True, parents=True)

load_dotenv(find_dotenv(usecwd=True))
API_KEY = os.environ.get("MP_API_KEY")
assert API_KEY, "MP_API_KEY not found in .env"
print("python", sys.version.split()[0], "| data folder:", DATA_DIR)





python 3.11.17 | data folder: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data


In [29]:
FIELDS = ["formula_pretty", "elements", "nelements", "nsites", "volume", "density", "symmetry", "formation_energy_per_atom", "energy_above_hull", "is_stable", "band_gap", "vbm", "cbm",
         ]
with MPRester(API_KEY) as mpr:
    try:
        db_version = mpr.get_database_version()
    except Exception as e:
        db_version = "Unknown"
    docs = mpr.materials.summary.search(
        band_gap = (MIN_GAP, MAX_GAP),
        fields=FIELDS,
        chunk_size=CHUNK_SIZE,
        num_chunks=1 if TEST_RUN else None,
    )
print(f"Downloaded {len(docs):,} materials | MP database version: {db_version}")


C:\Users\solom\AppData\Local\Temp\ipykernel_3964\2430029740.py:5: MPRestWarning: `get_database_version` has been deprecated in favor of MPRester().db_version.
  db_version = mpr.get_database_version()
Retrieving SummaryDoc documents: 100%|██████████| 76824/76824 [01:05<00:00, 1175.00it/s]

Downloaded 76,824 materials | MP database version: 2026.09.28


In [30]:
def to_row(d):
    """Sumarry of Document of MP into a plan dict"""
    sym = getattr(d, "symmetry", None)
    crystal_system = getattr(d, "crystal_system", None) if sym else None
    elements = getattr(d, "crystal_system", None) if sym else None
    return{
        "formula": getattr(d, "formula_pretty", None),
        "elements": "_".join(sorted(str(e) for e in elements)) if elements else None,
        "nelements": getattr(d, "nelements", None),
        "nsites": getattr(d, "nsites", None),
        "volume": getattr(d, "volume", None),
        "density": getattr(d, "density", None),
        "crystal_system": getattr(d, "crystal_system", None),
        "space_group": getattr(d, "space_ group", None),
        "formation_energy_per_atom": getattr(d, "formation_energy_per_atom", None),
        "energy_above_hull": getattr(d, "energy_above_hull", None),
        "is_stable": getattr(d, "is_stable", None),
        "band_gap": getattr(d, "band_gap", None),
        "cbm": getattr(d, "cbm", None),
        "vbm": getattr(d, "vbm", None),
        "is_gap_direct": getattr(d, "is_gap_direct", None),
        
    }

df = pd.DataFrame([to_row(d) for d in docs])
df.head()

,formula,elements,nelements,nsites,volume,density,crystal_system,space_group,formation_energy_per_atom,energy_above_hull,is_stable,band_gap,cbm,vbm,is_gap_direct
0,Ac2O3,None,2,5,91.511224,9.109130,None,None,-3.737668,0.000000,True,3.5219,8.1120,4.5901,None
1,Ac2S3,None,2,40,1118.407852,6.535149,None,None,-2.493064,0.000000,True,2.2729,7.0869,4.8140,None
2,Ac2S3,None,2,10,328.464893,5.562971,None,None,-2.440364,0.052700,False,3.0275,5.9344,2.9069,None
3,AcAgTe2,None,3,4,122.518406,7.997421,None,None,-0.996232,0.253251,False,0.0748,6.0799,6.0051,None
4,AcAlO3,None,3,5,57.451413,8.728230,None,None,-3.690019,0.000000,True,4.1024,10.9060,6.8036,None


In [31]:
out_name = "RawData_test.csv" if TEST_RUN else "RawData.csv"
out_path = DATA_DIR / out_name
df.to_csv(out_path, index=False)

meta = {

    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "file_name": out_name,
    "rows": int(len(df)),
    "rows_with_cbm_and_vbm": int(df[["cbm", "vbm"]].notna().all(axis=1).sum()),
    "band_gap_window_eV": [MIN_GAP, MAX_GAP],
    "test_run": TEST_RUN,
    "mp_database_version": db_version,
    "mp_api_version": version("mp-api"),
}
(DATA_DIR / f"{out_path.stem}_metadata.json")
print("Saved:", out_path)
print(json.dumps(meta, indent=2))

Saved: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\RawData.csv
{
  "saved_at": "2026-10-20T01:28:40",
  "file_name": "RawData.csv",
  "rows": 76824,
  "rows_with_cbm_and_vbm": 76824,
  "band_gap_window_eV": [
    0.01,
    5.0
  ],
  "test_run": false,
  "mp_database_version": "2026.09.28",
  "mp_api_version": "0.46.5"
}
